# Preparation

In [6]:
#@title Import Libs
#@markdown All libraries import goes over here

import os
import glob
import joblib
import subprocess
import time
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [7]:
#@title Import Datasets

USE_MY_DRIVE = True #@param {type:"boolean"}
#@markdown > If you decided to **USE_MY_DRIVE** make sure to first run the latest version of [**02_data_preparation.ipynb**](https://colab.research.google.com/drive/1mdcqA8y9RN-pgtv3hRnDm2ZsheJqQnSe?usp=drive_link) with save to drive on
SHARED_FOLDER_URL = 'https://drive.google.com/drive/folders/1oj1cqdfyh1ouXThXJbjgamLA6H5nX8nF'

data_dir = 'processed_data'
loaded_from_my_drive = False

if USE_MY_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = '/content/drive/MyDrive/DataScience-ProjectUAS/processed_data'
        if os.path.exists(f'{drive_path}/dataset_1_splits.pkl'):
            data_dir = drive_path
            loaded_from_my_drive = True
            print(f"Loaded from personal Google Drive: {data_dir}")
    except ImportError:
        pass

if not loaded_from_my_drive:
    os.makedirs(data_dir, exist_ok=True)
    if not glob.glob(f'{data_dir}/**/dataset_1_splits.pkl', recursive=True):
        print("Downloading splits from shared Google Drive folder...")
        subprocess.run(
            ['gdown', '--folder', SHARED_FOLDER_URL, '-O', data_dir, '--remaining-ok'],
            check=True
        )
    matched = glob.glob(f'{data_dir}/**/dataset_1_splits.pkl', recursive=True)
    data_dir = os.path.dirname(matched[0])
    print(f"Loaded from shared Drive fallback: {data_dir}")

dataset_1 = joblib.load(f'{data_dir}/dataset_1_splits.pkl')
dataset_2 = joblib.load(f'{data_dir}/dataset_2_splits.pkl')
dataset_3 = joblib.load(f'{data_dir}/dataset_3_splits.pkl')

print("All datasets ready for modeling.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded from shared Drive fallback: processed_data
All datasets ready for modeling.


### Checking Dataset

In [8]:
from typing import Required
datasets = {
    "Dataset 1": dataset_1,
    "Dataset 2": dataset_2,
    "Dataset 3": dataset_3
}

required_keys = ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"]

for name, data in datasets.items():
  missing_keys = [key for key in required_keys if key not in data]
  if missing_keys:
    raise KeyError(f"{name} belum memiliki key: {missing_keys}")

  print(f"{name}")
  print(f"  Train     : X={data['X_train'].shape}, y={data['y_train'].shape}")
  print(f"  Validation: X={data['X_val'].shape}, y={data['y_val'].shape}")
  print(f"  Test      : X={data['X_test'].shape}, y={data['y_test'].shape}")

Dataset 1
  Train     : X=(18152, 116), y=(18152,)
  Validation: X=(3890, 116), y=(3890,)
  Test      : X=(3890, 116), y=(3890,)
Dataset 2
  Train     : X=(699838, 27), y=(699838,)
  Validation: X=(149965, 27), y=(149965,)
  Test      : X=(149966, 27), y=(149966,)
Dataset 3
  Train     : X=(7000, 28), y=(7000,)
  Validation: X=(1500, 28), y=(1500,)
  Test      : X=(1500, 28), y=(1500,)


### Menentukan Strategi dan Ruang Pencarian

In [11]:

model_pipeline = Pipeline([
    ("model", LinearRegression())
])


param_grid = [
    {
        "model": [LinearRegression()],
        "model__fit_intercept": [True, False]
    },
    {
        "model": [Ridge()],
        "model__alpha": [0.01, 0.1, 1, 10, 100],
        "model__fit_intercept": [True, False]
    }
]

cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

N_JOBS = 1

print("Jumlah konfigurasi model:", 2 + (5 * 2))
print("Jumlah fold:", cv_strategy.get_n_splits())


Jumlah konfigurasi model: 12
Jumlah fold: 5


### Fungsi Validasi Baseline & Tuning

In [12]:
def calculate_metrics(model, X_train, y_train, X_val, y_val):
    """Menghitung metrik pada train dan validation."""
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    return {
        "Train_RMSE": float(np.sqrt(mean_squared_error(y_train, train_pred))),
        "Val_RMSE": float(np.sqrt(mean_squared_error(y_val, val_pred))),
        "Val_MAE": float(mean_absolute_error(y_val, val_pred)),
        "Val_R2": float(r2_score(y_val, val_pred))
    }


def evaluate_original_baseline(data, dataset_name):
    """Membuat metrik pembanding menggunakan Linear Regression default."""
    baseline_model = LinearRegression()
    baseline_model.fit(data["X_train"], data["y_train"])

    metrics = calculate_metrics(
        baseline_model,
        data["X_train"], data["y_train"],
        data["X_val"], data["y_val"]
    )
    metrics["Dataset"] = dataset_name
    metrics["Baseline_Model"] = "LinearRegression"
    return baseline_model, metrics


def tune_baseline(data, dataset_name):
    """Melakukan Grid Search 5-fold CV pada data training."""
    search = GridSearchCV(
        estimator=model_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv_strategy,
        n_jobs=N_JOBS,
        refit=True,
        return_train_score=True,
        pre_dispatch="n_jobs",
        error_score="raise"
    )

    start_time = time.time()
    search.fit(data["X_train"], data["y_train"])
    tuning_time = time.time() - start_time

    best_model = search.best_estimator_
    metrics = calculate_metrics(
        best_model,
        data["X_train"], data["y_train"],
        data["X_val"], data["y_val"]
    )

    best_index = search.best_index_
    cv_train_rmse = -float(search.cv_results_["mean_train_score"][best_index])
    cv_val_rmse = -float(search.best_score_)

    result = {
        "Dataset": dataset_name,
        "Tuned_Model": type(best_model.named_steps["model"]).__name__,
        "Best_Params": search.best_params_,
        "Best_CV_RMSE": cv_val_rmse,
        "CV_Train_RMSE": cv_train_rmse,
        "CV_Gap": cv_val_rmse - cv_train_rmse,
        "Tuned_Train_RMSE": metrics["Train_RMSE"],
        "Tuned_Val_RMSE": metrics["Val_RMSE"],
        "Tuned_Val_MAE": metrics["Val_MAE"],
        "Tuned_Val_R2": metrics["Val_R2"],
        "Tuning_Time_Sec": tuning_time
    }
    return search, result


### Tuning Datasets

In [13]:
baseline_models = {}
baseline_results = []
tuning_searches = {}
tuning_results = []

for dataset_name, data in datasets.items():
    print(f"\nMemproses {dataset_name}...")

    baseline_model, baseline_result = evaluate_original_baseline(data, dataset_name)
    baseline_models[dataset_name] = baseline_model
    baseline_results.append(baseline_result)

    search, tuning_result = tune_baseline(data, dataset_name)
    tuning_searches[dataset_name] = search
    tuning_results.append(tuning_result)

    print("Model terbaik:", tuning_result["Tuned_Model"])
    print("Parameter terbaik:", tuning_result["Best_Params"])
    print(f"Best CV RMSE: {tuning_result['Best_CV_RMSE']:.4f}")
    print(f"Validation RMSE setelah tuning: {tuning_result['Tuned_Val_RMSE']:.4f}")

baseline_metrics_df = pd.DataFrame(baseline_results)
tuning_results_df = pd.DataFrame(tuning_results)

print("\nTuning untuk seluruh dataset selesai.")
display(tuning_results_df)



Memproses Dataset 1...
Model terbaik: Ridge
Parameter terbaik: {'model': Ridge(), 'model__alpha': 1, 'model__fit_intercept': True}
Best CV RMSE: 42059.7261
Validation RMSE setelah tuning: 43526.5087

Memproses Dataset 2...
Model terbaik: Ridge
Parameter terbaik: {'model': Ridge(), 'model__alpha': 1, 'model__fit_intercept': True}
Best CV RMSE: 0.5001
Validation RMSE setelah tuning: 0.4999

Memproses Dataset 3...
Model terbaik: Ridge
Parameter terbaik: {'model': Ridge(), 'model__alpha': 1, 'model__fit_intercept': True}
Best CV RMSE: 5.0196
Validation RMSE setelah tuning: 5.0935

Tuning untuk seluruh dataset selesai.


,Dataset,Tuned_Model,Best_Params,Best_CV_RMSE,CV_Train_RMSE,CV_Gap,Tuned_Train_RMSE,Tuned_Val_RMSE,Tuned_Val_MAE,Tuned_Val_R2,Tuning_Time_Sec
0,Dataset 1,Ridge,"{'model': Ridge(), 'model__alpha': 1, 'model__...",42059.726120,41721.698581,338.027539,41754.800295,43526.508712,30215.042807,0.747202,3.883588
1,Dataset 2,Ridge,"{'model': Ridge(), 'model__alpha': 1, 'model__...",0.500109,0.500089,0.000020,0.500091,0.499890,0.398886,0.913032,30.033534
2,Dataset 3,Ridge,"{'model': Ridge(), 'model__alpha': 1, 'model__...",5.019583,4.999314,0.020269,5.001497,5.093486,4.059713,0.982172,0.612510


### Membandingkan Sebelum & Sesudah

In [14]:
comparison_df = baseline_metrics_df[[
    "Dataset", "Baseline_Model", "Train_RMSE", "Val_RMSE", "Val_MAE", "Val_R2"
]].merge(
    tuning_results_df[[
        "Dataset", "Tuned_Model", "Tuned_Train_RMSE", "Tuned_Val_RMSE",
        "Tuned_Val_MAE", "Tuned_Val_R2", "Best_CV_RMSE", "Best_Params"
    ]],
    on="Dataset",
    how="inner"
)

comparison_df["RMSE_Improvement_Pct"] = (
    (comparison_df["Val_RMSE"] - comparison_df["Tuned_Val_RMSE"])
    / comparison_df["Val_RMSE"] * 100
)

comparison_df = comparison_df.rename(columns={
    "Train_RMSE": "Baseline_Train_RMSE",
    "Val_RMSE": "Baseline_Val_RMSE",
    "Val_MAE": "Baseline_Val_MAE",
    "Val_R2": "Baseline_Val_R2"
})

display(comparison_df)


,Dataset,Baseline_Model,Baseline_Train_RMSE,Baseline_Val_RMSE,Baseline_Val_MAE,Baseline_Val_R2,Tuned_Model,Tuned_Train_RMSE,Tuned_Val_RMSE,Tuned_Val_MAE,Tuned_Val_R2,Best_CV_RMSE,Best_Params,RMSE_Improvement_Pct
0,Dataset 1,LinearRegression,41750.642970,43524.454474,30270.268427,0.747226,Ridge,41754.800295,43526.508712,30215.042807,0.747202,42059.726120,"{'model': Ridge(), 'model__alpha': 1, 'model__...",-4.719732e-03
1,Dataset 2,LinearRegression,0.500091,0.499890,0.398886,0.913032,Ridge,0.500091,0.499890,0.398886,0.913032,0.500109,"{'model': Ridge(), 'model__alpha': 1, 'model__...",4.772755e-07
2,Dataset 3,LinearRegression,5.001490,5.093535,4.059889,0.982171,Ridge,5.001497,5.093486,4.059713,0.982172,5.019583,"{'model': Ridge(), 'model__alpha': 1, 'model__...",9.470587e-04


### Hasil Cross-Validation

In [15]:
cv_result_list = []

for dataset_name, search in tuning_searches.items():
    cv_df = pd.DataFrame(search.cv_results_).copy()
    cv_df["Dataset"] = dataset_name
    cv_df["Mean_Train_RMSE"] = -cv_df["mean_train_score"]
    cv_df["Mean_CV_RMSE"] = -cv_df["mean_test_score"]
    cv_result_list.append(cv_df)

cv_results_df = pd.concat(cv_result_list, ignore_index=True)
cv_results_view = cv_results_df[[
    "Dataset", "params", "Mean_Train_RMSE", "Mean_CV_RMSE",
    "std_test_score", "rank_test_score"
]].rename(columns={"std_test_score": "CV_Score_Std", "rank_test_score": "Rank"})

cv_results_view = cv_results_view.sort_values(["Dataset", "Rank"])
display(cv_results_view)


,Dataset,params,Mean_Train_RMSE,Mean_CV_RMSE,CV_Score_Std,Rank
6,Dataset 1,"{'model': Ridge(), 'model__alpha': 1, 'model__...",41721.698581,42059.726120,665.671705,1
5,Dataset 1,"{'model': Ridge(), 'model__alpha': 0.1, 'model...",41715.476837,42060.462943,666.016021,2
4,Dataset 1,"{'model': Ridge(), 'model__alpha': 0.1, 'model...",41715.465154,42060.468687,666.071278,3
7,Dataset 1,"{'model': Ridge(), 'model__alpha': 1, 'model__...",41722.634409,42060.506384,665.171416,4
3,Dataset 1,"{'model': Ridge(), 'model__alpha': 0.01, 'mode...",41715.390897,42061.126139,666.156180,5
2,Dataset 1,"{'model': Ridge(), 'model__alpha': 0.01, 'mode...",41715.390778,42061.127783,666.161772,6
1,Dataset 1,"{'model': LinearRegression(), 'model__fit_inte...",41715.390012,42061.208951,666.172517,7
0,Dataset 1,"{'model': LinearRegression(), 'model__fit_inte...",41715.390012,42061.208951,666.172517,8
8,Dataset 1,"{'model': Ridge(), 'model__alpha': 10, 'model_...",41949.033242,42238.479324,674.439900,9
9,Dataset 1,"{'model': Ridge(), 'model__alpha': 10, 'model_...",41976.914471,42265.667409,670.464300,10


### Simpan hasil tuning

In [16]:
output_dir = os.path.join(os.path.dirname(data_dir) or ".", "tuning_baseline_outputs")
model_dir = os.path.join(output_dir, "models")
result_dir = os.path.join(output_dir, "results")

os.makedirs(model_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

for dataset_name, search in tuning_searches.items():
    dataset_number = dataset_name.split()[-1]
    model_path = os.path.join(model_dir, f"tuned_baseline_dataset_{dataset_number}.pkl")
    joblib.dump(search.best_estimator_, model_path)
    print(f"Model {dataset_name} disimpan: {model_path}")

baseline_metrics_df.to_csv(
    os.path.join(result_dir, "baseline_original_metrics.csv"), index=False
)
tuning_results_df.to_csv(
    os.path.join(result_dir, "baseline_tuning_results.csv"), index=False
)
comparison_df.to_csv(
    os.path.join(result_dir, "baseline_comparison_results.csv"), index=False
)
cv_results_view.to_csv(
    os.path.join(result_dir, "baseline_cv_results.csv"), index=False
)

print(f"\nSeluruh model dan hasil tuning tersimpan di: {output_dir}")


Model Dataset 1 disimpan: ./tuning_baseline_outputs/models/tuned_baseline_dataset_1.pkl
Model Dataset 2 disimpan: ./tuning_baseline_outputs/models/tuned_baseline_dataset_2.pkl
Model Dataset 3 disimpan: ./tuning_baseline_outputs/models/tuned_baseline_dataset_3.pkl

Seluruh model dan hasil tuning tersimpan di: ./tuning_baseline_outputs
